# Module 6 — Customer Segmentation
Reuses `src/clustering.py` (built on the Module 5 engineered dataset).

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd() / "src"))
from clustering import *


## Load & Validate

In [2]:
df = load_feature_dataset()
validation = validate_dataset(df)
print(df.shape)
validation


(2237, 52)


,Check,Result
0,No missing values,True
1,Duplicate rows (post ID removal),184
2,All columns numeric,True
3,No identifier columns present,True


## Feature Selection & Scaling

In [3]:
X = select_features(df)
X_std, standard_scaler = apply_standard_scaler(X)
X_mm, minmax_scaler = apply_minmax_scaler(X)

save_scaler(standard_scaler, MODELS_DIR / "standard_scaler.pkl")
save_scaler(minmax_scaler, MODELS_DIR / "minmax_scaler.pkl")
X_std.shape


(2237, 45)

## Scaling Comparison

In [4]:
scaling_comparison = pd.DataFrame({
    "StandardScaler_mean": X_std.mean(),
    "StandardScaler_std": X_std.std(),
    "MinMaxScaler_min": X_mm.min(),
    "MinMaxScaler_max": X_mm.max(),
})
scaling_comparison.head(10)


,StandardScaler_mean,StandardScaler_std,MinMaxScaler_min,MinMaxScaler_max
Income,-2.223424e-17,1.000224,0.0,1.0
Kidhome,1.429344e-17,1.000224,0.0,1.0
Teenhome,5.399744e-17,1.000224,0.0,1.0
Recency,-3.851288e-17,1.000224,0.0,1.0
MntWines,-1.429344e-17,1.000224,0.0,1.0
MntFruits,3.176320e-18,1.000224,0.0,1.0
MntMeatProducts,-3.811584e-17,1.000224,0.0,1.0
MntFishProducts,3.970400e-17,1.000224,0.0,1.0
MntSweetProducts,2.699872e-17,1.000224,0.0,1.0
MntGoldProds,7.940799e-19,1.000224,0.0,1.0


StandardScaler is used going forward: it centers every feature at 0 with unit variance, which suits the distance-based algorithms below (K-Means, GMM). MinMaxScaler is fitted and saved for reference but not used for clustering, since it would leave outliers compressed into a narrow range without equalizing variance across features.

## EDA for Clustering

In [5]:
X_std.describe().T.head(10)


,count,mean,std,min,25%,50%,75%,max
Income,2237.0,-2.223424e-17,1.000224,-2.394700,-0.780248,-0.022612,0.784757,3.132264
Kidhome,2237.0,1.429344e-17,1.000224,-0.825388,-0.825388,-0.825388,1.032151,2.889690
Teenhome,2237.0,5.399744e-17,1.000224,-0.930227,-0.930227,-0.930227,0.906417,2.743062
Recency,2237.0,-3.851288e-17,1.000224,-1.696210,-0.867183,-0.003613,0.859956,1.723526
MntWines,2237.0,-1.429344e-17,1.000224,-2.592263,-0.807232,0.271874,0.859572,1.461068
MntFruits,2237.0,3.176320e-18,1.000224,-1.426634,-0.984954,-0.026539,0.820400,1.949510
MntMeatProducts,2237.0,-3.811584e-17,1.000224,-2.646166,-0.829836,0.058895,0.848411,2.132198
MntFishProducts,2237.0,3.970400e-17,1.000224,-1.529067,-0.692641,0.018504,0.843214,1.825990
MntSweetProducts,2237.0,2.699872e-17,1.000224,-1.405668,-0.970709,-0.026881,0.807170,2.093314
MntGoldProds,2237.0,7.940799e-19,1.000224,-2.424663,-0.635789,0.076076,0.716376,2.154688


## Elbow Method

In [6]:
elbow = elbow_method(X_std, range(2, 11), save_path=FIGURES_DIR / "elbow.png")
elbow


,k,inertia
0,2,81152.708162
1,3,74800.035878
2,4,71430.855881
3,5,68773.853017
4,6,66675.197975
5,7,64379.881185
6,8,62226.845707
7,9,60809.204062
8,10,59743.668963


## Silhouette Analysis

In [7]:
sil = silhouette_analysis(X_std, range(2, 11), save_path=FIGURES_DIR / "silhouette.png")
sil


,k,silhouette_score
0,2,0.183366
1,3,0.129831
2,4,0.107835
3,5,0.110928
4,6,0.100352
5,7,0.121970
6,8,0.116820
7,9,0.111773
8,10,0.102100


## Davies-Bouldin Analysis

In [8]:
db = davies_bouldin_analysis(X_std, range(2, 11))
db


,k,davies_bouldin_score
0,2,1.965531
1,3,2.276374
2,4,2.612599
3,5,2.474912
4,6,2.543835
5,7,2.123232
6,8,2.398584
7,9,2.381219
8,10,2.489303


## Chosen k
Selected from the elbow/silhouette/Davies-Bouldin results above.

In [9]:
best_k = int(sil.loc[sil["silhouette_score"].idxmax(), "k"])
best_k


2

## Train Models

In [10]:
kmeans_model, kmeans_labels = train_kmeans(X_std, best_k)
hier_model, hier_labels = train_hierarchical(X_std, best_k, save_path=FIGURES_DIR / "dendrogram.png")
gmm_model, gmm_labels = train_gmm(X_std, best_k)
dbscan_model, dbscan_labels = train_dbscan(X_std, eps=2.5, min_samples=10)


## Model Comparison

In [11]:
comparison = compare_models(X_std, {
    "KMeans": kmeans_labels,
    "Hierarchical": hier_labels,
    "GMM": gmm_labels,
    "DBSCAN": dbscan_labels,
})
comparison


,model,n_clusters,silhouette_score,davies_bouldin_score
0,KMeans,2,0.183366,1.965531
1,Hierarchical,2,0.160066,2.132937
2,GMM,2,0.184529,2.609888
3,DBSCAN,2,-0.100540,1.575272


## Final Model Selection

In [12]:
comparison.sort_values("silhouette_score", ascending=False)


,model,n_clusters,silhouette_score,davies_bouldin_score
2,GMM,2,0.184529,2.609888
0,KMeans,2,0.183366,1.965531
1,Hierarchical,2,0.160066,2.132937
3,DBSCAN,2,-0.100540,1.575272


K-Means is retained as the final segmentation model. On some runs another algorithm (e.g. GMM) may edge it out on silhouette score, but the metrics above are close, and K-Means gives every customer a single hard cluster assignment (unlike GMM's soft assignments and DBSCAN's noise points), which is simpler to deploy and act on. This also matches the labels already used for profiling and visualization below, and the model saved into the production pipeline.

## Cluster Profiling
Using KMeans labels (selected model).

In [13]:
profile = profile_clusters(df, kmeans_labels)
cluster_summary = generate_cluster_summary(df, kmeans_labels)
profile


,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,...,Marital_Status_Widow,Preferred_Shopping_Channel_Catalog,Preferred_Shopping_Channel_Store,Preferred_Shopping_Channel_Web,Product_Preference_Fish,Product_Preference_Fruits,Product_Preference_Gold,Product_Preference_Meat,Product_Preference_Sweets,Product_Preference_Wine
Cluster,,,,,,,,,,,,,,,,,,,,,
0,-0.791348,0.688621,-0.023285,-0.008421,-0.865196,-0.661071,-0.865900,-0.663476,-0.646694,-0.641349,...,0.023832,0.002860,0.734986,0.262154,0.046711,0.014299,0.146806,0.197331,0.013346,0.581506
1,0.698757,-0.608050,0.020561,0.007436,0.763965,0.583724,0.764587,0.585847,0.571029,0.566309,...,0.043771,0.111953,0.613636,0.274411,0.008418,0.000000,0.010943,0.198653,0.001684,0.780303


## Visualizations

In [14]:
visualize_pca(X_std, kmeans_labels, save_path=FIGURES_DIR / "pca.png")
visualize_tsne(X_std, kmeans_labels, save_path=FIGURES_DIR / "tsne.png")
cluster_heatmap(profile, save_path=FIGURES_DIR / "heatmap.png")
radar_features = ["Income", "Total_Spending", "Total_Purchases", "Recency", "Age"]
radar_chart(profile, radar_features, save_path=FIGURES_DIR / "radar.png")


## Stability Analysis

In [15]:
stability = stability_analysis(X_std, best_k)
stability


,pairwise_ari,mean_ari
0,0.996425,0.995713
1,0.996425,0.995713
2,0.996425,0.995713
3,0.996425,0.995713
4,0.992857,0.995713
5,1.000000,0.995713
6,0.992857,0.995713
7,0.992857,0.995713
8,1.000000,0.995713
9,0.992857,0.995713


## Save Models & Pipeline

In [16]:
save_model(kmeans_model, MODELS_DIR / "kmeans.pkl")
save_model(hier_model, MODELS_DIR / "hierarchical.pkl")
save_model(gmm_model, MODELS_DIR / "gmm.pkl")
save_model(dbscan_model, MODELS_DIR / "dbscan.pkl")
save_final_pipeline(standard_scaler, kmeans_model, list(X.columns), MODELS_DIR / "pipeline.pkl")


PosixPath('/home/xploit/Downloads/ML-Projects/Day-6/outputs/models/pipeline.pkl')

In [17]:
selected_features = list(X.columns)
joblib.dump(selected_features, MODELS_DIR / "selected_features.pkl")
print(f"Saved: {MODELS_DIR / 'selected_features.pkl'}")


Saved: /home/xploit/Downloads/ML-Projects/Day-6/outputs/models/selected_features.pkl


## Export Reports

In [18]:
export_reports({
    "algorithm_comparison": comparison,
    "cluster_profiles": profile,
    "cluster_summary": cluster_summary,
    "metrics": db,
})


[PosixPath('/home/xploit/Downloads/ML-Projects/Day-6/outputs/reports/algorithm_comparison.xlsx'),
 PosixPath('/home/xploit/Downloads/ML-Projects/Day-6/outputs/reports/cluster_profiles.xlsx'),
 PosixPath('/home/xploit/Downloads/ML-Projects/Day-6/outputs/reports/cluster_summary.xlsx'),
 PosixPath('/home/xploit/Downloads/ML-Projects/Day-6/outputs/reports/metrics.xlsx')]

## Conclusion
KMeans (k = best_k, selected via silhouette score) was chosen as the final segmentation model. Models, scalers, the deployable pipeline, and all reports/figures are saved under `outputs/`.